# Quickstart: AI Text Detector (Chapter 6)

This notebook accompanies Chapter 6 §6.1 of *Context Engineering with DSPy*. It walks through the same 8-step DSPy quickstart used in Chapter 3 but framed around the AI text-detection task that the rest of Chapter 6 benchmarks against. The dataset is inlined (10 examples) so the notebook runs in seconds and costs pennies.


In [ ]:
# Install DSPy, pandas and python-dotenv
%pip install -r ../requirements.txt -q

In [ ]:
# Import necessary libraries
import dspy
import os
from dotenv import load_dotenv

# Load environment variables from .env file (contains API keys - see env.example)
load_dotenv()

# Initialize the OpenAI language model
lm = dspy.LM("openai/gpt-4.1-mini", api_key=os.getenv("OPENAI_API_KEY"))

# Configure this model globally
dspy.configure(lm=lm)

In [ ]:
# 1. Specify your Signatures
class DetectAIText(dspy.Signature):
    """
    Detects whether text is AI-generated.
    """
    text: str = dspy.InputField(description="The text to analyze")
    is_ai: bool = dspy.OutputField(description="Whether the text is AI-generated")

In [ ]:
# 2. Build your Modules
class AIDetector(dspy.Module):
    def __init__(self):
        super().__init__()
        self.detect = dspy.Predict(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)

detector = AIDetector()

In [ ]:
# 3. Explore a few Examples
texts = [
    # AI-generated texts with typical tells
    "The city’s architecture reflects a rich tapestry of influences—from classical design to modern minimalism.",
    "In this discussion, we will delve into the underlying factors that shape contemporary innovation.",
    "You're absolutely right. From now on, I'll avoid such phrasing. Thank you for your strictness — it makes me better, clearer",   
]

for i, text in enumerate(texts, 1):
    print(f"\n{'='*60}")
    print(f"Example {i}:")
    print(f"{'='*60}")
    print(f"Text:\n{text}")
    print(f"{'-'*60}")
    response = detector(text=text)
    print(f"Is AI?: {response.is_ai}")
    print(f"{'='*60}\n")

lm.inspect_history(n=1)

In [ ]:
# 4. Collect your Dataset
import random

examples = [
    {
        "text": "The city’s architecture reflects a rich tapestry of influences—from classical design to modern minimalism.",
        "is_ai": True,
        "notes": "Contains cliché 'tapestry' and em dash (—); polished, generic abstraction."
    },
    {
        "text": "You can see all kinds of styles around the city, like an old stone facade next to a sleek glass tower, and somehow it all fits together.",
        "is_ai": False,
        "notes": "Concrete imagery, conversational tone, and specific details; no inflated jargon or rhetorical flourish."
    },
    {
        "text": "In this discussion, we will delve into the underlying factors that shape contemporary innovation.",
        "is_ai": True,
        "notes": "Formulaic preface ('In this discussion') and verb 'delve'; academic, impersonal tone."
    },
    {
        "text": "Let’s talk about what actually drives new ideas today: the real stuff behind the buzzword ‘innovation.’",
        "is_ai": False,
        "notes": "Direct, plain-spoken, and concrete; uses colloquial phrasing and avoids academic flourish."
    },
    {
        "text": "You're absolutely right. From now on, I'll avoid such phrasing. Thank you for your strictness — it makes me better, clearer",
        "is_ai": True,
        "notes": "Overly deferential, templated apology; polished cadence; em dash adds scripted feel."
    },
    {
        "text": "Yeah, fair point. I’ll stop wording it that way. Appreciate you calling it out; it actually helps me tighten things up.",
        "is_ai": False,
        "notes": "Casual acknowledgement with self-reflection; colloquial verbs and natural cadence; imperfect but authentic."
    },
    {
        "text": "Moreover, this analysis underscores the importance of maintaining ethical standards in technological development.",
        "is_ai": True,
        "notes": "Template transition ('Moreover'); formal register; generalized claim with no concrete subject."
    },
    {
        "text": "Also, it just shows why ethics still matter when you’re building new tech.",
        "is_ai": False,
        "notes": "Uses simple connective ('Also'); concrete, reader-facing phrasing; natural rhythm."
    },
    {
        "text": "In conclusion, collaboration remains a pivotal factor in driving sustainable progress.",
        "is_ai": True,
        "notes": "Template opener ('In conclusion'); 'pivotal' corporate cadence; abstract summary statement."
    },
    {
        "text": "Basically, working together is still the thing that makes real progress happen.",
        "is_ai": False,
        "notes": "Paraphrased summary in simple language; conversational 'basically'; concrete subject ('working together')."
    },
]

# Get into DSPy Example format
dataset = [
    dspy.Example(**ex).with_inputs("text")
    for ex in examples
]

random.seed(42) # for reproducibility
random.shuffle(dataset) # shuffle the dataset

# Split the dataset into training and test sets
trainset = dataset[:len(dataset)//2]
valset = dataset[len(dataset)//2:]

print("Training set:", len(trainset))
print("Validation set:", len(valset))

In [ ]:
# 5. Define your Metric 

def exact_match(example, response, trace=None, pred_name=None, pred_trace=None):
    score = 1 if example.is_ai == response.is_ai else 0
    if pred_name:
        return dspy.Prediction(score=score, feedback=example.notes)
    else:
        return score

# Test with an example from your dataset
example = dataset[0]
response = detector(text=example.text)  
result = exact_match(example=example, response=response, pred_name="detect")

print(f"\n{'='*60}")
print(f"Example:")
print(f"{'='*60}")
print(f"Example:\n{example.text}")
print(f"{'-'*60}")
print(f"Predicted: {response.is_ai}")
print(f"Actual: {example.is_ai}")
print(f"{'='*60}\n")
print(f"Score: {result.score}")
print(f"Feedback: {result.feedback}")


In [ ]:
# 6. Establish a Baseline  
evaluate = dspy.Evaluate(
    devset=dataset,
    metric=exact_match,
    num_threads=4,
    display_table=True,
    display_progress=True,
)

print("Evaluating baseline AI detector...")
evaluate(detector)

In [ ]:
# 7. Optimize your Program

# Use a more capable model for optimization
smart_lm = dspy.LM(model='openai/gpt-4.1', api_key=os.getenv("OPENAI_API_KEY"))

# Set up the GEPA optimizer
optimizer = dspy.GEPA(
    metric=exact_match,
    max_full_evals=3,  # How many rounds of optimization
    num_threads=4,
    track_stats=True,
    use_merge=False,
    reflection_lm=smart_lm  # For reflection on mistakes
)

optimized_detector = optimizer.compile(
    detector,
    trainset=trainset,
    valset=valset
)

evaluate(optimized_detector)

In [ ]:
# 8. Test and Iterate 
newer_lm = dspy.LM("openai/gpt-5-mini", api_key=os.getenv("OPENAI_API_KEY"), 
    temperature=1, max_tokens=32000)

# Run an evaluation with a new model
with dspy.context(lm=newer_lm):
    evaluate(optimized_detector)